In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [8]:
df['gender'].describe()

count        100
unique         2
top       Female
freq          59
Name: gender, dtype: object

In [10]:
df['cough'].describe()

count      100
unique       2
top       Mild
freq        62
Name: cough, dtype: object

In [11]:
from sklearn.model_selection import train_test_split

In [13]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=0)

In [14]:
x_train

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


# Without column transfer

In [24]:
si = SimpleImputer() # Because it had null values
x_train_fever = si.fit_transform(x_train[['fever']])

x_test_fever = si.transform(x_test[['fever']])

type(x_train_fever)

numpy.ndarray

In [26]:
# Oridnal Encoding on ordinal values

oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

x_test_cough = oe.transform(x_test[['cough']])

In [34]:
# One Hot Encoding

ohe = OneHotEncoder(drop='first', sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender', 'city']])
x_test_gender_city = ohe.fit_transform(x_test[['gender', 'city']])


x_train_gender_city

In [41]:
# Extracting Age

x_train_age = x_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
type(x_train_age)

x_test_age = x_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [45]:
x_train_transformed = np.concatenate([
    x_train_age,
    x_train_gender_city,
    x_train_fever,
    x_train_cough
], axis=1)

x_test_transformed = np.concatenate([
    x_test_age,
    x_test_gender_city,
    x_test_fever,
    x_test_cough
], axis=1)

In [46]:
x_train_transformed.shape

(80, 7)

# With Column transform

In [49]:
from sklearn.compose import ColumnTransformer

In [62]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [63]:
type(transformer.fit_transform(x_train))

numpy.ndarray

In [64]:
transformer.transform(x_test)

array([[100.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  19.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  25.        ],
       [101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  81.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   5.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  27.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  69.        ],
       [ 98.        ,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  34.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          0.    